**0. Imports:**

In [ ]:
import earthcarekit as eck
eck.set_config("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
import xarray as xr
xr.set_options(display_expand_data_vars=True, display_max_rows=500, display_max_items=10000)
import calendar
import re
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path
from datetime import datetime, timedelta, date
import os
from scipy.interpolate import interp1d
import sys
import shlex
import shutil

**0b. Remote ZIP Helpers**

In [ ]:
MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))

from my_subprocess import run_shell_cmd_and_communicate, print_shell_output


def _to_date(v):
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(f"Unsupported date type: {type(v)}")


def run_cmd_checked(cmd: str, verbose: bool = False):
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=verbose)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err


def discover_remote_zip_files(remote_product_root, start_date, end_date):
    root  = Path(remote_product_root)
    start = _to_date(start_date)
    end   = _to_date(end_date)
    if end < start:
        raise ValueError("end_date must be >= start_date")
    zips = []
    day  = start
    while day <= end:
        day_dir = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if day_dir.exists():
            zips.extend(sorted(day_dir.glob("*.ZIP")))
            zips.extend(sorted(day_dir.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))


def stage_zip_and_extract(src_zip: Path, local_stage_root: Path):
    local_stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = local_stage_root / src_zip.name
    extract_dir = local_stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No .h5 after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files


def cleanup_staged_data(local_zip, extract_dir):
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()

**1. Config**

In [ ]:
REMOTE_PRODUCT_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_TC__2A")

LOCAL_STAGE_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_2A")
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)

# Full date range — the monthly loop iterates over each calendar month within this range
FULL_START = date(2025, 1, 1)
FULL_END   = date(2026, 2, 28)

# Output: all monthly files go into ATC_output/monthly/
OUTPUT_ROOT    = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output")
MONTHLY_OUTDIR = OUTPUT_ROOT / "monthly"
MONTHLY_OUTDIR.mkdir(parents=True, exist_ok=True)

CLASS_VAR    = "classification"
TARGET_CLASS = 3      # Ice Cloud
GROUND_CODE  = -2     # Surface or sub-surface (excluded from denominator)
MISSING_CODE = -3     # Missing Data (excluded from denominator)

# tc=-1 (noise in both Mie and Ray channels): excluded when True
EXCLUDE_NOISE_CLASS = True

QC_VAR      = "quality_status"
MAX_QC_FLAG = 1    # accept 0 (Good) and 1 (Likely Good); reject 2, 3, 4

LAT        = "latitude"
LON        = "longitude"
HEIGHT_VAR = "height"

GRID_RES_DEG         = 1.0
MAX_HEIGHT_M         = 20_000.0
MIN_SAMPLES_PER_CELL = 10

if not REMOTE_PRODUCT_ROOT.exists():
    raise FileNotFoundError(f"Remote path not mounted/reachable: {REMOTE_PRODUCT_ROOT}")

print("Config loaded")
print(f"Full range:     {FULL_START} -> {FULL_END}")
print(f"Monthly output: {MONTHLY_OUTDIR}")
print(f"TARGET_CLASS:   {TARGET_CLASS}  GROUND_CODE={GROUND_CODE}  MISSING_CODE={MISSING_CODE}")
print(f"EXCLUDE_NOISE_CLASS (tc=-1): {EXCLUDE_NOISE_CLASS}")
print(f"MAX_QC_FLAG={MAX_QC_FLAG}  MIN_SAMPLES_PER_CELL={MIN_SAMPLES_PER_CELL}")

**2. Grid Setup**

In [ ]:
lat_bins = np.arange(-90.0, 90.0 + GRID_RES_DEG, GRID_RES_DEG)
lon_bins = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)

lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
lon_centers = 0.5 * (lon_bins[:-1] + lon_bins[1:])

n_lat = len(lat_centers)
n_lon = len(lon_centers)

target_h = np.arange(0.0, MAX_HEIGHT_M + 1.0, 100.0)   # 201 levels: 0..20000 m
n_height  = target_h.size

print("Grid ready")
print(f"Shape (lat, lon, height): ({n_lat}, {n_lon}, {n_height})")
print(f"Height range: {float(np.nanmin(target_h)):.0f} .. {float(np.nanmax(target_h)):.0f} m")

**3. One-File Accumulator**

In [ ]:
def accumulate_one_file(fp, lat_bins, lon_bins, target_h):
    with eck.read_product(str(fp)) as ds:
        cls = ds[CLASS_VAR].values.astype(float)
        h   = ds[HEIGHT_VAR].values
        lat = ds[LAT].values
        lon = ds[LON].values
        qc  = ds[QC_VAR].values.astype(int)

        n_obs    = cls.shape[0]
        n_height = target_h.size
        ice_interp = np.full((n_obs, n_height), np.nan, dtype=float)

        for i in range(n_obs):
            h_i   = h[i, :]
            cls_i = cls[i, :]
            qc_i  = qc[i, :]

            valid = (np.isfinite(h_i) & np.isfinite(cls_i)
                     & (cls_i != GROUND_CODE)
                     & (cls_i != MISSING_CODE)
                     & (qc_i  <= MAX_QC_FLAG))
            if EXCLUDE_NOISE_CLASS:
                valid &= (cls_i != -1)
            if np.sum(valid) < 2:
                continue

            h_v   = h_i[valid]
            ice_v = (cls_i[valid] == TARGET_CLASS).astype(float)

            order = np.argsort(h_v)
            h_v, ice_v = h_v[order], ice_v[order]
            h_v, idx   = np.unique(h_v, return_index=True)
            ice_v      = ice_v[idx]
            if h_v.size < 2:
                continue

            ice_interp[i, :] = interp1d(h_v, ice_v, kind="nearest",
                                         bounds_error=False, fill_value=np.nan)(target_h)

    total_hist_3d = np.zeros((len(lat_bins)-1, len(lon_bins)-1, n_height), dtype=np.float64)
    ice_hist_3d   = np.zeros_like(total_hist_3d)

    lat2d = np.broadcast_to(lat[:, None], ice_interp.shape)
    lon2d = np.broadcast_to(lon[:, None], ice_interp.shape)

    for k in range(n_height):
        v = ice_interp[:, k]
        m = np.isfinite(v) & np.isfinite(lat2d[:, k]) & np.isfinite(lon2d[:, k])
        if not np.any(m):
            continue
        total_k, _, _ = np.histogram2d(lat2d[:, k][m], lon2d[:, k][m],
                                        bins=[lat_bins, lon_bins])
        ice_k, _, _   = np.histogram2d(lat2d[:, k][m], lon2d[:, k][m],
                                        bins=[lat_bins, lon_bins],
                                        weights=(v[m] == 1).astype(float))
        total_hist_3d[:, :, k] = total_k
        ice_hist_3d[:, :, k]   = ice_k

    return total_hist_3d, ice_hist_3d


print("accumulate_one_file() defined")

**4. Monthly Processing Loop**

Iterates over every calendar month from `FULL_START` to `FULL_END`.
For each month: discovers ZIPs, accumulates counts, computes occurrence, saves three NetCDF files to `ATC_output/monthly/`.

In [ ]:
def _month_end(d):
    """Last day of the month containing date d."""
    last_day = calendar.monthrange(d.year, d.month)[1]
    return date(d.year, d.month, last_day)


def _next_month(d):
    """First day of the month after date d."""
    if d.month == 12:
        return date(d.year + 1, 1, 1)
    return date(d.year, d.month + 1, 1)


month_start = FULL_START

while month_start <= FULL_END:
    month_end = min(_month_end(month_start), FULL_END)

    print(f"\n{'='*60}")
    print(f"Processing {month_start:%B %Y}  ({month_start} -> {month_end})")
    print(f"{'='*60}")

    # Skip if output files already exist (safe to re-run)
    _base  = f"ATL_TC_2A_{GRID_RES_DEG}deg_{month_start:%Y%m%d}_{month_end:%Y%m%d}"
    _check = MONTHLY_OUTDIR / f"{_base}_occurrence_latheight.nc"
    if _check.exists():
        print(f"  Already done — skipping ({_check.name})")
        month_start = _next_month(month_start)
        continue

    # Discover ZIPs for this month
    zip_paths_month = discover_remote_zip_files(REMOTE_PRODUCT_ROOT, month_start, month_end)
    if not zip_paths_month:
        print("  No ZIPs found — skipping")
        month_start = _next_month(month_start)
        continue
    print(f"  Found {len(zip_paths_month)} ZIPs")

    # Fresh accumulators
    total_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
    ice_count_3d   = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)

    failed_zips  = []
    processed_h5 = 0
    n_zips_month = len(zip_paths_month)

    for idx, src_zip in enumerate(zip_paths_month, start=1):
        local_zip = None
        extract_dir = None
        try:
            local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
            for fp in h5_files:
                t3d, i3d = accumulate_one_file(fp, lat_bins, lon_bins, target_h)
                total_count_3d += t3d
                ice_count_3d   += i3d
                processed_h5   += 1
        except Exception as e:
            failed_zips.append((str(src_zip), str(e)))
        finally:
            cleanup_staged_data(local_zip, extract_dir)

        if idx % 200 == 0 or idx == n_zips_month:
            print(f"  {idx}/{n_zips_month} ZIPs | h5={processed_h5} | failed={len(failed_zips)}")

    print(f"  Batch done: {processed_h5} H5 files, {len(failed_zips)} failures")
    if failed_zips:
        print(f"  First failure: {failed_zips[0][1]}")

    # --- Derive occurrence maps ---

    # Lat/lon (height-collapsed)
    ice_2d = np.nansum(ice_count_3d,   axis=2)
    tot_2d = np.nansum(total_count_3d, axis=2)
    occurrence_2d = np.divide(ice_2d, tot_2d,
                               out=np.full_like(ice_2d, np.nan), where=tot_2d > 0)
    occurrence_2d[tot_2d < MIN_SAMPLES_PER_CELL] = np.nan

    # Lat/height (longitude-collapsed)
    ice_lh = np.nansum(ice_count_3d,   axis=1)
    tot_lh = np.nansum(total_count_3d, axis=1)
    occurrence_lh = np.divide(ice_lh, tot_lh,
                               out=np.full_like(ice_lh, np.nan), where=tot_lh > 0)
    occurrence_lh[tot_lh < MIN_SAMPLES_PER_CELL] = np.nan

    # 3D (lat x lon x height)
    occurrence_3d = np.divide(ice_count_3d, total_count_3d,
                               out=np.full_like(ice_count_3d, np.nan), where=total_count_3d > 0)
    occurrence_3d[total_count_3d < MIN_SAMPLES_PER_CELL] = np.nan

    tot_sum = np.nansum(total_count_3d)
    if tot_sum > 0:
        print(f"  Global ice occurrence: {float(np.nansum(ice_count_3d) / tot_sum):.4f}")

    # --- Save ---
    attrs_base = {"source": "EarthCARE ATL_TC__2A",
                  "month": f"{month_start:%Y-%m}",
                  "target_class": TARGET_CLASS,
                  "exclude_noise_class": str(EXCLUDE_NOISE_CLASS),
                  "max_qc_flag": MAX_QC_FLAG,
                  "grid_resolution_deg": GRID_RES_DEG,
                  "min_samples_per_cell": MIN_SAMPLES_PER_CELL}

    xr.Dataset(
        {"occurrence":  (["latitude", "longitude", "height"], occurrence_3d),
         "ice_count":   (["latitude", "longitude", "height"], ice_count_3d),
         "total_count": (["latitude", "longitude", "height"], total_count_3d)},
        coords={"latitude": lat_centers, "longitude": lon_centers, "height": target_h},
        attrs=attrs_base,
    ).to_netcdf(MONTHLY_OUTDIR / f"{_base}_occurrence_3d.nc")

    xr.Dataset(
        {"occurrence":  (["latitude", "longitude"], occurrence_2d),
         "ice_count":   (["latitude", "longitude"], ice_2d),
         "total_count": (["latitude", "longitude"], tot_2d)},
        coords={"latitude": lat_centers, "longitude": lon_centers},
        attrs=attrs_base,
    ).to_netcdf(MONTHLY_OUTDIR / f"{_base}_occurrence_latlon.nc")

    xr.Dataset(
        {"occurrence":  (["latitude", "height"], occurrence_lh),
         "ice_count":   (["latitude", "height"], ice_lh),
         "total_count": (["latitude", "height"], tot_lh)},
        coords={"latitude": lat_centers, "height": target_h},
        attrs=attrs_base,
    ).to_netcdf(MONTHLY_OUTDIR / f"{_base}_occurrence_latheight.nc")

    print(f"  Saved: {_base}_occurrence_[3d|latlon|latheight].nc")

    month_start = _next_month(month_start)

print("\nAll months complete.")
print(f"Output directory: {MONTHLY_OUTDIR}")